In [1]:
import os
import cv2
import random
from collections import defaultdict

# =========================
# USER INPUT PATHS
# =========================

source_path = r"/mnt/lustre/lalit47/us_data/bench/STMUS"
target_path = r"/mnt/lustre/lalit47/down/class/STMUS"

train_ratio = 0.7
val_ratio = 0.1
test_ratio = 0.2

random_seed = 42
random.seed(random_seed)

# Only these main folders will be used
selected_folders = ["TA", "BB", "GM"]

# Only these class folders will be used
class_folders = ["Healthy", "Pathological"]


def process_and_save(src, dst):
    im = cv2.imread(src, cv2.IMREAD_GRAYSCALE)

    if im is None:
        print(f"Failed to read: {src}")
        return False

    im = cv2.resize(im, (256, 256), interpolation=cv2.INTER_LINEAR)
    cv2.imwrite(dst, im)
    return True


def is_version_1_image(file_name):
    """
    Select only images ending with _1.png, for example:
    anon_2_1.png
    anon2_1.png
    """

    file_name_lower = file_name.lower()

    if not file_name_lower.endswith(".png"):
        return False

    base_name = os.path.splitext(file_name_lower)[0]

    return base_name.endswith("_1")


def collect_classification_images():
    data_by_class = defaultdict(list)

    for main_folder in selected_folders:
        main_dir = os.path.join(source_path, main_folder)

        if not os.path.isdir(main_dir):
            print(f"Main folder not found: {main_dir}")
            continue

        for class_name in class_folders:
            images_dir = os.path.join(main_dir, class_name, "Images")

            if not os.path.isdir(images_dir):
                print(f"Images folder not found: {images_dir}")
                continue

            for file_name in os.listdir(images_dir):

                if not is_version_1_image(file_name):
                    continue

                image_path = os.path.join(images_dir, file_name)
                base_name = os.path.splitext(file_name)[0]

                # Class label will be Healthy or Pathological
                data_by_class[class_name].append(
                    (image_path, base_name, class_name, main_folder)
                )

    return data_by_class


def stratified_split(data_by_class):
    train_data = []
    val_data = []
    test_data = []

    for class_name, samples in data_by_class.items():
        random.shuffle(samples)

        n = len(samples)
        n_train = int(n * train_ratio)
        n_val = int(n * val_ratio)

        train_samples = samples[:n_train]
        val_samples = samples[n_train:n_train + n_val]
        test_samples = samples[n_train + n_val:]

        train_data.extend(train_samples)
        val_data.extend(val_samples)
        test_data.extend(test_samples)

        print(
            f"{class_name}: total={n}, "
            f"train={len(train_samples)}, "
            f"val={len(val_samples)}, "
            f"test={len(test_samples)}"
        )

    return train_data, val_data, test_data


def save_classification_split(data, split_name):
    for image_path, base_name, class_name, main_folder in data:
        class_out_dir = os.path.join(target_path, split_name, class_name)
        os.makedirs(class_out_dir, exist_ok=True)

        # Add main folder name to avoid filename conflicts
        image_dst = os.path.join(
            class_out_dir,
            f"{main_folder}_{base_name}.png"
        )

        process_and_save(image_path, image_dst)


data_by_class = collect_classification_images()

train_data, val_data, test_data = stratified_split(data_by_class)

save_classification_split(train_data, "train")
save_classification_split(val_data, "val")
save_classification_split(test_data, "test")

print("\nClassification dataset created successfully!")

Healthy: total=2206, train=1544, val=220, test=442
Pathological: total=447, train=312, val=44, test=91

Classification dataset created successfully!
